## Ionic Liquid Selection and Reactor-Separator Network Optimization 

This script is a Julia implementation of ilrs_MIP.ipynb. 
It uses the JuMP package for optimization and the Gurobi solver to find the optimal solution. The script defines a mixed-integer nonlinear programming (MINLP) model to select ionic liquid pair and reactor-separator network while minimizing cost. 

Discretization of the nonlinear term and conversion to quadratic unconstrained binary optimization model (QUBO) are explored in this script.  

The problem is adapted from the following source:

    Iftakher, A., & Hasan, M. M. F. (2024). Exploring quantum optimization for computer-aided Molecular and Process Design. Systems and Control Transactions, 3, 292–299. https://psecommunity.org/LAPSE:2024.1540

### Formulation in the original paper (P8) 

#### Definitions
<div style="font-size:80%">

- $ \text{Cat} = \{ c_1, c_2, \dots, c_C \} $: set of cations  
- $ \text{An} = \{ a_1, a_2, \dots, a_A \} $: set of anions  
- $ \text{IL} = \{ c \times a \mid c \in \text{Cat},\, a \in \text{An} \} $: set of all feasible ionic liquids (ILs)  
- $ K $: set of all process units  
- $ K_r \subset K $: set of all reactors  
- $ K_s \subset K $: set of all separators  
- $ \alpha_k $: stoichiometric reactor conversion factor for reactor $ k \in K_r $  
- $ \beta_{k, c, a} $: separation factor for separator $ k \in K_s $ with IL constructed by cation $ c $ and anion $ a $  
- $ I_k^{\text{in}} $: set of all inlet streams to unit $ k $  
- $ I_k^{\text{out}} $: set of all outlet streams from unit $ k $  
- $ d $: demand requirement  

**Decision Variables**  
- $ x_i $: flowrate of stream $ i $  
- $ z_c $: binary variable indicating selection of cation $ c $  
- $ z_a $: binary variable indicating selection of anion $ a $  
- $ y_k $: binary variable indicating selection of process unit $ k $  

**Cost Coefficients**  
- $ c_k^f $: fixed cost for unit $ k $  
- $ c_i^I $: operating cost for reactor stream $ i \in I_{Kr}^{\text{in}} $  
- $ c_k^I $: operating cost associated with separator $ k $  
- $ c_k^e $: cost associated with emissions or waste for separator $ k $
</div>

### Formulation
<div style="font-size:80%">

$$
\begin{aligned}
    & \min && \sum_{k \in K} c_k^f y_k 
    + \sum_{i \in I_{Kr}^{in}} c_i^I x_i^{0.6} 
    + \sum_{k \in K_s} c_k^I \left( \sum_{i \in I_{Ks}^{in}} x_i \right)^2 \\
    & && + \sum_{k \in K_s} c_k^e \left( \sum_{i \in I_{Ks}^{in}} x_i - \sum_{i \in I_{Ks}^{out}} x_i \right) \\
    \\
    & \text{s.t.} && f_k^L y_k \leq \sum_{i \in I_k^{in}} x_i \leq f_k^U y_k, \quad \forall k \in K \\
    & && \sum_{i \in I_{Kr}^{out}} x_i = \alpha_k \sum_{i \in I_{Kr}^{in}} x_i, \quad \forall k \in K_r \\
    & && \sum_{c \in Cat} z_c = 1 \\
    & && \sum_{a \in An} z_a = 1 \\
    & && x_i \geq \beta_{k, c, a} \sum_{i \in I_{Ks}^{in}} x_i - M (2 - z_c - z_a),  \quad \forall i \in I_{Ks}^{out}, \, k \in K_s, \, c \in Cat, \, a \in An \\
    & && x_i \leq \beta_{k, c, a} \sum_{i \in I_{Ks}^{in}} x_i + M (2 - z_c - z_a),  \quad \forall i \in I_{Ks}^{out}, \, k \in K_s, \, c \in Cat, \, a \in An \\
    & && \sum_{i \in I_{Ks}^{out}} x_i \geq d
\end{aligned}
$$

</div>

In [1]:
using JuMP
using QUBO
using Plots
using SparseArrays
using Gurobi
using CSV
using DataFrames
using DWave

include("../ilrs_utils.jl")

const data_filepath = joinpath(@__DIR__, "..", "data")


/home/yirangpark/repos/secquoia/pd_ising/il-rxtor-sep-opt/.CondaPkg/env/lib/python3.10/site-packages/dwave/cloud/utils.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import iter_entry_points
┌ Warning: The 'DWAVE_API_TOKEN' environment variable is not defined.
│ If you want to use D-Wave's cloud services, please make sure that another access method is available.
│ 
│ For more information visit:
│     https://docs.ocean.dwavesys.com/en/stable/overview/sapi.html
└ @ DWave /home/yirangpark/.julia/packages/DWave/67a0J/src/DWave.jl:23


"/home/yirangpark/repos/secquoia/pd_ising/il-rxtor-sep-opt/original_mip/../data"

In [2]:
# === Build the model without discretization ===
m1 = build_base_model_mip(;disc=false)

# === Solve using Gurobi and display the results ===
solve_and_display(m1)


Model is built without discretization.
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-21
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 13th Gen Intel(R) Core(TM) i7-1365U, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 34 rows, 22 columns and 148 nonzeros
Model fingerprint: 0x8a9c8f98
Model has 1 general nonlinear constraint (11 nonlinear terms)
Variable types: 13 continuous, 9 integer (9 binary)
Coefficient statistics:
  Matrix range     [6e-01, 1e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [2e+00, 2e+00]
  RHS range        [1e+00, 2e+03]
Presolve model has 1 nlconstr
Added 19 variables to disaggregate expressions.
Presolve removed 2 rows and 2 columns
Presolve time: 0.01s
Presolved: 81 rows, 40 columns, 261 nonzeros
Presolved model has 9 bilinear constraint(s)
Presolved model has 2 nonlinear constr

## Conversion into QUBO using QUBO.jl and discretization 

QUBO.jl is unable to convert ^0.6 terms in the objective automatically, so x variables are discretized prior to conversion.
 

In [3]:
m_disc = build_base_model_mip(;disc=true)

# === Solve using Gurobi and display the results ===
solve_and_display(m_disc)


Model (and objective function) is built with discretization of x.
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-21
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 13th Gen Intel(R) Core(TM) i7-1365U, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 38 rows, 62 columns and 232 nonzeros
Model fingerprint: 0x4ab851ea
Model has 9 quadratic objective terms
Variable types: 11 continuous, 51 integer (51 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+03]
  Objective range  [1e-01, 2e+01]
  QObjective range [4e-01, 3e+00]
  Bounds range     [2e+00, 2e+00]
  RHS range        [1e+00, 2e+03]
Found heuristic solution: objective 55.3170916
Presolve removed 2 rows and 4 columns
Presolve time: 0.00s
Presolved: 36 rows, 58 columns, 266 nonzeros
Presolved model has 9 quadratic objective terms
Variable types: 9 continuou

In [4]:
m_disc_1 = build_base_model_mip(;disc=true)

# === Convert to QUBO ===
dwave_optimizer = ToQUBO.Optimizer(DWave.Neal.Optimizer, num_reads=100, num_sweeps=3000)

# The continuous variables x is discretized automatically by QUBO.jl (other than the ^0.6 expression that was discretized as specified in build_base_model)
m_qubo_disc = convert_to_qubo(identity, m_disc_1; optimizer=dwave_optimizer)
println(m_qubo_disc)

# Print the results
solution_summary(m_disc_1)

Model (and objective function) is built with discretization of x.
QUBOTools Model
▷ Sense ………………… Min
▷ Domain ……………… BoolDomain
▷ Variables ……… 438

Density:
▷ Linear ……………… 100.00%
▷ Quadratic ………   6.21%
▷ Total …………………   6.43%

There are no warm-start values.

There are no solutions available.



solution_summary(; result = 1, verbose = false)
├ solver_name          : Virtual QUBO Model
├ Termination
│ ├ termination_status : LOCALLY_SOLVED
│ ├ result_count       : 1000
│ └ raw_status         : 
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : NO_SOLUTION
│ └ objective_value      : 2.27540e+04
└ Work counters
  └ solve_time (sec)   : 5.51117e+00

## Conversion into QUBO using QUBO.jl and encoding method--Ifthaker.jl 

QUBO.jl is unable to convert ^0.6 terms in the objective automatically, so an encoding method is introduced to discretize x variables. 

In [6]:
include("ifthaker.jl")

@doc"""
    discretize_model!(model::JuMP.Model; J::Int=1, variables = [])
Discretize the model by setting the discretization attribute for each variable using the encoding method "ifthaker.jl"
    Args:
        model: The JuMP model to be discretized.
        J: The number of decimal places to keep (default is 1).
        variables: A vector of variable names to be discretized (default is empty).

    Returns:
        None
"""
function discretize_model!(model::JuMP.Model; J::Int=1, variables = [])
    # disable the automatic discretization in ToQUBO
    set_attribute(model, ToQUBO.Attributes.Discretize(), false)
    
    # Loop over all variables passed in
    for v in variables
        if v isa JuMP.VariableRef
            # Single variable
            set_attribute(v, ToQUBO.Attributes.VariableEncodingMethod(), Ifthaker())
            set_attribute(v, ToQUBO.Attributes.VariableEncodingBits(), 4 * J + 1)
        elseif v isa AbstractArray
            # JuMP container (array, DenseAxisArray, etc.)
            for subv in v
                set_attribute(subv, ToQUBO.Attributes.VariableEncodingMethod(), Ifthaker())
                set_attribute(subv, ToQUBO.Attributes.VariableEncodingBits(), 4 * J + 1)
            end
        else
            error("Unsupported variable type: $v")
        end
    end

    # define the tolerance for the encoding of the continuous variable (x1)
    # set_attribute(x1, ToQUBO.Attributes.VariableEncodingATol(), 0.05)

    # similarly, define the encoding method and number of bits for the slack variable for the constraint (c1)
    # set_attribute(c1, ToQUBO.Attributes.SlackVariableEncodingMethod(), Ifthaker())
    # set_attribute(c1, ToQUBO.Attributes.SlackVariableEncodingBits(), 4J + 1)
end


discretize_model!

In [ ]:
m_disc_2 = build_base_model_mip(;disc=true)

# === Convert to QUBO ===
dwave_optimizer = ToQUBO.Optimizer(DWave.Neal.Optimizer, num_reads=1000, num_sweeps=3000)

# The continuous variables x is discretized using ifthaker encoding method (other than the ^0.6 expression that was discretized as specified in build_base_model)
m_qubo_disc_2 = convert_to_qubo(
    model -> discretize_model!(model; J = 1, variables = model[:x]), 
    m_disc_2; 
    optimizer = dwave_optimizer
    )

println(m_qubo_disc_2)

# Print the results
solution_summary(m_disc_2)

# Display the results
println("Objective value: ", objective_value(m_disc_2))
for var in all_variables(m_disc_2)
    println("$(name(var)) = ", value(var))
end

Model (and objective function) is built with discretization of x.
QUBOTools Model
▷ Sense ………………… Min
▷ Domain ……………… BoolDomain
▷ Variables ……… 383

Density:
▷ Linear ……………… 100.00%
▷ Quadratic ………  10.24%
▷ Total …………………  10.47%

There are no warm-start values.

There are no solutions available.

Objective value: 243739.4306640625
y[r1] = 1.0
y[r2] = 1.0
y[s1] = 1.0
y[s2] = 1.0
y[s3] = 0.0
z_cat[1] = 0.0
z_cat[2] = 0.0
z_an[1] = 1.0
z_an[2] = 1.0
x[("srce", "r1")] = 2.0
x[("r1", "s1")] = 1.4000000000000001
x[("r1", "s2")] = 0.0
x[("r1", "s3")] = 0.6000000000000001
x[("srce", "r2")] = 2.0000000000000004
x[("r2", "s1")] = 2.0
x[("r2", "s2")] = 0.0
x[("r2", "s3")] = 0.0
x[("s1", "sink")] = 0.0
x[("s2", "sink")] = 0.0
x[("s3", "sink")] = 0.6000000000000001
k[1,1] = 0.0
k[2,1] = 0.0
k[1,2] = 0.0
k[2,2] = 0.0
k[1,3] = 0.0
k[2,3] = 0.0
k[1,4] = 0.0
k[2,4] = 0.0
k[1,5] = 0.0
k[2,5] = 0.0
k[1,6] = 0.0
k[2,6] = 0.0
k[1,7] = 0.0
k[2,7] = 0.0
k[1,8] = 0.0
k[2,8] = 1.0
k[1,9] = 1.0
k[2,9] = 0.0
k

In [ ]:
m_gurobi = build_base_model_mip(;disc=true)

gurobi_optimizer = ToQUBO.Optimizer(Gurobi.Optimizer)

# === Convert to QUBO ===
# dwave_optimizer = ToQUBO.Optimizer(DWave.Neal.Optimizer, num_reads=1000, num_sweeps=3000)
m_qubo_gurobi, m_qubo_mapping = convert_to_qubo(identity, m_gurobi; optimizer=gurobi_optimizer)
# m_qubo_gurobi = convert_to_qubo(identity, m_gurobi)